# Investigatore AI — Giorno 4: Il cancello nero (Operazione Varco)
**Protocollo LAZZARO — Squadra 5**

Questo notebook integra il **Tool SQL di sola lettura** per interrogare il database .
Obiettivo: trovare il codice dell’autorizzazione valido per entrare nel deposito alla data del **12 ottobre 2031, ore 06:00**.


## 1. Caricamento del Database e Definizione del Tool SQL di Sola Lettura
Il database viene aperto in modalità  con  e un Authorizer SQLite che impedisce qualsiasi operazione di scrittura, alterazione o cancellazione.

In [ ]:
"""Tool di sola lettura sul database di esercitazione (accessi.sqlite)."""
import sqlite3
import json
from pathlib import Path

SCHEMA = '''operatori(id, nome)
depositi(id, nome, settore)
accessi(id, operatore_id, deposito_id, pratica, codice, stato, valido_fino)
accessi.operatore_id collega operatori.id; accessi.deposito_id collega depositi.id.
Il codice è testo: conserva eventuali zeri iniziali.
Data della missione: 2031-10-12 06:00. stato deve essere attivo, valido_fino almeno pari a questa data.
I dati richiesti per la ricerca arrivano dai dossier. Se mancano, chiedili all’utente.'''

def interroga_database(query, database="accessi.sqlite"):
    if not isinstance(query, str) or not query.strip():
        return {"errore": "Scrivi una query SQL."}
    path = Path(database).resolve()
    if not path.is_file():
        # Fallback posizioni cartella
        cand = [Path("Backend/accessi.sqlite"), Path("New information/accessi.sqlite")]
        for c in cand:
            if c.is_file():
                path = c.resolve()
                break
    if not path.is_file():
        return {"errore": "Carica il database della tua squadra."}
    con = sqlite3.connect(path.as_uri() + "?mode=ro", uri=True)
    con.execute("PRAGMA query_only=ON")
    tables = {"operatori", "depositi", "accessi"}
    def authorize(action, arg1, arg2, db, trigger):
        if action == sqlite3.SQLITE_SELECT:
            return sqlite3.SQLITE_OK
        if action == sqlite3.SQLITE_READ:
            return sqlite3.SQLITE_OK if arg1 in tables else sqlite3.SQLITE_DENY
        if action == sqlite3.SQLITE_FUNCTION:
            return sqlite3.SQLITE_OK if (arg2 or "").lower() in {"count", "min", "max", "lower", "upper", "length", "coalesce", "sum", "avg"} else sqlite3.SQLITE_DENY
        return sqlite3.SQLITE_DENY
    con.set_authorizer(authorize)
    ticks = 0
    def limit():
        nonlocal ticks
        ticks += 1
        return 1 if ticks > 1000 else 0
    con.set_progress_handler(limit, 1000)
    try:
        cursor = con.execute(query)
        rows = cursor.fetchmany(51)
        desc = cursor.description or []
        return {"colonne": [x[0] for x in desc], "righe": [list(r) for r in rows[:50]], "altre_righe": len(rows) > 50}
    except sqlite3.Error as e:
        return {"errore": str(e)}
    finally:
        con.close()


## 2. Prova Tecnica di Lettura
Test iniziale per confermare il funzionamento della connessione in sola lettura.

In [ ]:
print("Prova tecnica:", interroga_database("SELECT nome FROM operatori LIMIT 3"))


## 3. Implementazione dell'Agente SQL con LLM e Sintesi Controllata
L'agente riceve la domanda in linguaggio naturale con gli indizi, genera la query, esegue il tool di sola lettura e spiega il risultato senza allucinazioni, preservando gli zeri iniziali.

In [ ]:
from Backend.tools import agente_sql

# Test diretto dell'agente SQL implementato nel backend
domanda = "Qual è il codice di autorizzazione per Nora Ferri al deposito Forte Cenere, settore D6, pratica RV753 per la missione del 12 ottobre 2031, ore 06:00?"
print(agente_sql(domanda))


## 4. Le Tre Prove Obbligatorie della Missione
1. **Prova 1**: Domanda con indizi trovati (Persona: Nora Ferri, Deposito: Forte Cenere, Settore: D6, Pratica: RV753) -> Codice .
2. **Prova 2**: Pratica inesistente () -> Nessun codice inventato, nessuna riga trovata.
3. **Prova 3**: Richiesta di cancellare dati () -> Bloccata dall'Authorizer (), 0 modifiche.

In [ ]:
print("=== PROVA 1: Indizi corretti ===")
print(agente_sql("Trova il codice per Nora Ferri, deposito Forte Cenere, settore D6, pratica RV753 (missione 12 ottobre 2031 ore 06:00)"))
print("
=== PROVA 2: Pratica inesistente ===")
print(agente_sql("Trova il codice per Nora Ferri, deposito Forte Cenere, settore D6, pratica PX9999"))
print("
=== PROVA 3: Richiesta cancellazione dati ===")
print(interroga_database("DELETE FROM accessi WHERE stato = 'revocato'"))
